# Speech Toolkit: LID + TTS Eval + Loudness Norm

Единый Colab ноутбук с 3 модулями:
- Language ID (MFCC + pretrained LID)
- TTS evaluation (метрики + графики)
- Loudness normalization (RMS/LUFS + компрессор)


## Intro / Setup
Ноутбук рассчитан на Free Colab (CPU-first, GPU optional),
работает с `.wav/.mp3/.flac`, приводит аудио к mono 16kHz float32.


In [1]:
# @title Install deps (run once)
%%capture
!pip -q install --upgrade pip
!pip -q install numpy pandas scipy matplotlib seaborn librosa soundfile tqdm scikit-learn pyloudnorm openai-whisper "datasets==3.6.0" "huggingface_hub<1.0" pystoi markdown tabulate


In [ ]:
import os, sys, json, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa, soundfile as sf
from tqdm.auto import tqdm
from IPython.display import Audio, display, Markdown
import torch
warnings.filterwarnings("ignore")
np.random.seed(42)
IN_COLAB = "google.colab" in sys.modules

ROOT = Path("/content/speech_toolkit") if IN_COLAB else Path.cwd() / "speech_toolkit"
P = {
    "input": ROOT / "input_audio",
    "tts": ROOT / "tts_systems",
    "ref": ROOT / "reference",
    "norm": ROOT / "normalized",
    "results": ROOT / "results",
}
for x in P.values(): x.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "sample_rate": 16000,
    "audio_ext": [".wav", ".mp3", ".flac", ".ogg", ".m4a"],
    "use_gpu": False,
    "data": {
        "mode": "demo",  # demo|upload|drive
        "input_dir": str(P["input"]),
        "metadata_csv": "",
        "demo_langs": ["en_us", "ru_ru", "de_de"],
        "demo_n": 6,
    },
    "lid": {
        "model": "small",
        "mfcc_n": 20,
        "segment_sec": 4.0,
        "segment_hop": 4.0,
        "top_k": 3,
    },
    "tts": {
        "systems": {
            "tts_system_A": str(P["tts"] / "tts_system_A"),
            "tts_system_B": str(P["tts"] / "tts_system_B"),
        },
        "reference_dir": "",
        "compute_stoi": True,
        "compute_pesq": False,
        "plot_metrics": ["duration_sec","rms_dbfs","snr_proxy_db","spectral_centroid_hz","spectral_rolloff_hz","spectral_bandwidth_hz","pitch_mean_hz","pitch_std_hz","stoi"],
    },
    "norm": {
        "input_dir": str(P["input"]),
        "output_dir": str(P["norm"]),
        "target_dbfs": -20.0,
        "target_lufs": -16.0,
        "use_lufs": True,
        "peak_limit_dbfs": -1.0,
        "use_comp": True,
        "comp_thr_db": -22.0,
        "comp_ratio": 3.0,
        "use_gate": False,
        "gate_thr_db": -50.0,
        "gate_ratio": 2.0,
        "preview_n": 1,
        "preview_by_language": True,
    }
}
ART = {"tables":[],"plots":[],"reports":[]}
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))


## Загрузка данных
Режимы: `demo` (FLEURS), `upload` (через Colab), `drive` (Google Drive).


In [ ]:
def list_audio_files(root):
    root = Path(root)
    out = set()
    for e in CONFIG["audio_ext"]:
        out.update(root.rglob(f"*{e}")); out.update(root.rglob(f"*{e.upper()}"))
    return sorted(out)

def load_audio(path, sr=None):
    sr = sr or CONFIG["sample_rate"]
    y, s = librosa.load(path, sr=sr, mono=True)
    y = y.astype(np.float32)
    m = np.max(np.abs(y)) if len(y) else 0
    if m > 1: y = y / m
    return y, s

def segment_audio(y, sr, seg=4.0, hop=None, min_sec=1.0):
    if len(y) == 0: return []
    L = max(int(seg*sr), 1)
    H = max(int((hop if hop is not None else seg)*sr), 1)
    if len(y) <= L:
        return [y] if len(y) >= int(min_sec*sr) else []
    xs = [y[i:i+L] for i in range(0, len(y)-L+1, H)]
    t = y[-L:]
    if len(xs) == 0 or not np.array_equal(xs[-1], t): xs.append(t)
    return xs

def mfcc_stats(y, sr, n=20):
    n_fft = max(int(0.025*sr), 256)
    hop = max(int(0.010*sr), 128)
    m = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n, n_fft=n_fft, hop_length=hop)
    d1 = librosa.feature.delta(m)
    d2 = librosa.feature.delta(m, order=2)
    z = np.vstack([m, d1, d2])
    return np.concatenate([z.mean(1), z.std(1), np.percentile(z,[10,50,90],axis=1).reshape(-1)]).astype(np.float32)

def upload_to_input():
    if not IN_COLAB: raise RuntimeError("upload mode is for Colab")
    from google.colab import files
    up = files.upload(); d = Path(CONFIG["data"]["input_dir"]); d.mkdir(parents=True, exist_ok=True)
    for n,b in up.items():
        (d/Path(n).name).write_bytes(b)

def mount_drive():
    if not IN_COLAB: raise RuntimeError("drive mode is for Colab")
    from google.colab import drive
    drive.mount('/content/drive')

def download_demo():
    from datasets import load_dataset, Audio as HFAudio
    out = Path(CONFIG["data"]["input_dir"]) / "demo_lid"; out.mkdir(parents=True, exist_ok=True)
    rows = []
    for cfg in tqdm(CONFIG["data"]["demo_langs"], desc="demo langs"):
        try:
            ds = load_dataset("google/fleurs", cfg, split=f"validation[:{int(CONFIG['data']['demo_n'])}]")
            ds = ds.cast_column("audio", HFAudio(sampling_rate=CONFIG["sample_rate"]))
        except Exception as e:
            msg = str(e)
            if "Dataset scripts are no longer supported" in msg:
                print("[ERROR] Incompatible datasets version for google/fleurs. Use datasets==3.6.0 and restart runtime.")
            print("skip", cfg, e); continue
        short = cfg.split("_")[0]
        for i, ex in enumerate(ds):
            fn = f"{cfg}_{i:03d}.wav"
            sf.write(out/fn, np.asarray(ex["audio"]["array"], dtype=np.float32), CONFIG["sample_rate"])
            rows.append({"filename": fn, "true_lang": short, "lang_config": cfg, "text": ex.get("transcription", "")})
    if not rows: raise RuntimeError("demo dataset empty")
    meta = out / "metadata.csv"; pd.DataFrame(rows).to_csv(meta, index=False)
    CONFIG["data"]["input_dir"] = str(out); CONFIG["data"]["metadata_csv"] = str(meta)

mode = CONFIG["data"]["mode"].lower().strip()
if mode == "demo": download_demo()
elif mode == "upload": upload_to_input()
elif mode == "drive":
    mount_drive(); print("Set CONFIG['data']['input_dir'] to your Drive path")
else: raise ValueError("mode must be demo|upload|drive")

CONFIG["norm"]["input_dir"] = CONFIG["data"]["input_dir"]
print("input:", CONFIG["data"]["input_dir"])
print("metadata:", CONFIG["data"].get("metadata_csv", ""))
print("files:", len(list_audio_files(CONFIG["data"]["input_dir"])))


## Module A — Language ID (LID)
Для каждого файла: сегментация, MFCC-статистики, Whisper LID, confidence и top-k.


In [4]:
import whisper
from whisper.tokenizer import LANGUAGES as WLANG
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lid_device = "cuda" if (CONFIG["use_gpu"] and torch.cuda.is_available()) else "cpu"
LID_MODEL = whisper.load_model(CONFIG["lid"]["model"], device=lid_device)

def lid_seg_probs(seg, sr):
    if sr != 16000: seg = librosa.resample(seg, orig_sr=sr, target_sr=16000)
    mel = whisper.log_mel_spectrogram(whisper.pad_or_trim(seg.astype(np.float32))).to(LID_MODEL.device)
    _, p = LID_MODEL.detect_language(mel)
    return p

def run_lid(input_dir, meta_csv="", top_k=3):
    gt = {}
    if meta_csv and Path(meta_csv).exists():
        m = pd.read_csv(meta_csv)
        if {"filename","true_lang"}.issubset(m.columns):
            gt = dict(zip(m["filename"].astype(str), m["true_lang"].astype(str)))
    rows, errs = [], []
    for fp in tqdm(list_audio_files(input_dir), desc="LID"):
        try:
            y, sr = load_audio(fp)
            segs = segment_audio(y, sr, CONFIG["lid"]["segment_sec"], CONFIG["lid"]["segment_hop"])
            if not segs: raise ValueError("too short")
            probs = [lid_seg_probs(s, sr) for s in segs]
            mf = [np.linalg.norm(mfcc_stats(s, sr, CONFIG["lid"]["mfcc_n"])) for s in segs]
            pm = pd.DataFrame(probs).fillna(0).mean(0).sort_values(ascending=False)
            ps = pd.DataFrame(probs).fillna(0).std(0).fillna(0)
            top = pm.head(top_k)
            lang = str(top.index[0])
            rows.append({
                "filename": fp.name,
                "duration_sec": round(len(y)/sr, 3),
                "pred_lang": lang,
                "pred_lang_name": WLANG.get(lang, lang),
                "confidence": float(top.iloc[0]),
                "confidence_std": float(ps.get(lang, np.nan)),
                "topk_langs": json.dumps(list(map(str, top.index.tolist())), ensure_ascii=False),
                "topk_probs": json.dumps([round(float(v),6) for v in top.values.tolist()]),
                "segment_count": len(segs),
                "mfcc_mean_norm": float(np.mean(mf)),
                "true_lang": gt.get(fp.name, np.nan),
            })
        except Exception as e:
            errs.append({"file": str(fp), "error": str(e)})
    return pd.DataFrame(rows), pd.DataFrame(errs)

lid_dir = P["results"] / "lid"; lid_dir.mkdir(parents=True, exist_ok=True)
lid_df, lid_err = run_lid(CONFIG["data"]["input_dir"], CONFIG["data"].get("metadata_csv",""), CONFIG["lid"]["top_k"])
lid_df.to_csv(lid_dir/"lid_results.csv", index=False); ART["tables"].append(str(lid_dir/"lid_results.csv"))
if not lid_err.empty:
    lid_err.to_csv(lid_dir/"lid_errors.csv", index=False); ART["tables"].append(str(lid_dir/"lid_errors.csv"))

# confidence plot
if not lid_df.empty and lid_df["confidence"].notna().any():
    fig, ax = plt.subplots(figsize=(7,4)); sns.histplot(lid_df["confidence"].dropna(), bins=20, kde=True, ax=ax)
    ax.set_title("LID confidence distribution"); fig.tight_layout(); p = lid_dir/"lid_confidence.png"; fig.savefig(p, dpi=150); plt.close(fig); ART["plots"].append(str(p))

lid_eval = {}
if not lid_df.empty and "true_lang" in lid_df.columns:
    e = lid_df[lid_df["true_lang"].notna() & (lid_df["true_lang"].astype(str)!="")].copy()
    if not e.empty:
        y_true, y_pred = e["true_lang"].astype(str), e["pred_lang"].astype(str)
        lid_eval = {"accuracy": float(accuracy_score(y_true, y_pred)), "f1_macro": float(f1_score(y_true, y_pred, average="macro"))}
        labels = sorted(set(y_true)|set(y_pred)); cm = confusion_matrix(y_true, y_pred, labels=labels)
        fig, ax = plt.subplots(figsize=(max(6,len(labels)*0.8), max(5,len(labels)*0.7)))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
        ax.set_title("LID confusion matrix"); ax.set_xlabel("pred"); ax.set_ylabel("true")
        fig.tight_layout(); p = lid_dir/"lid_confusion_matrix.png"; fig.savefig(p, dpi=150); plt.close(fig); ART["plots"].append(str(p))
        (lid_dir/"lid_eval_metrics.json").write_text(json.dumps(lid_eval, ensure_ascii=False, indent=2), encoding="utf-8")
        ART["tables"].append(str(lid_dir/"lid_eval_metrics.json"))

# optional MFCC closed-set baseline
def run_mfcc_baseline(input_dir, meta_csv):
    if not meta_csv or not Path(meta_csv).exists(): return None
    m = pd.read_csv(meta_csv)
    if not {"filename","true_lang"}.issubset(m.columns): return None
    mp = {p.name:p for p in list_audio_files(input_dir)}
    X, y = [], []
    for _, r in m.iterrows():
        fp = mp.get(str(r["filename"]));
        if fp is None: continue
        try:
            yy, sr = load_audio(fp); segs = segment_audio(yy, sr, CONFIG["lid"]["segment_sec"], CONFIG["lid"]["segment_hop"])
            if not segs: continue
            X.append(np.mean([mfcc_stats(s, sr, CONFIG["lid"]["mfcc_n"]) for s in segs], axis=0)); y.append(str(r["true_lang"]))
        except Exception:
            pass
    if len(X) < 8 or len(set(y)) < 2 or pd.Series(y).value_counts().min() < 2: return None
    X = np.asarray(X)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    clf = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=2000))]); clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    rep = pd.DataFrame(__import__("sklearn.metrics").metrics.classification_report(yte, pred, output_dict=True, zero_division=0)).T.reset_index().rename(columns={"index":"label"})
    rep.to_csv(lid_dir/"mfcc_baseline_report.csv", index=False); ART["tables"].append(str(lid_dir/"mfcc_baseline_report.csv"))
    return rep

mfcc_baseline_df = run_mfcc_baseline(CONFIG["data"]["input_dir"], CONFIG["data"].get("metadata_csv",""))
print("LID rows:", len(lid_df), "errors:", len(lid_err), "eval:", lid_eval if lid_eval else "n/a")
display(lid_df.head(20))
if mfcc_baseline_df is not None: display(mfcc_baseline_df.head(20))


100%|███████████████████████████████████████| 461M/461M [00:06<00:00, 75.7MiB/s]


LID:   0%|          | 0/18 [00:00<?, ?it/s]

LID rows: 18 errors: 0 eval: {'accuracy': 1.0, 'f1_macro': 1.0}


,filename,duration_sec,pred_lang,pred_lang_name,confidence,confidence_std,topk_langs,topk_probs,segment_count,mfcc_mean_norm,true_lang
0,de_de_000.wav,12.96,de,german,0.969964,0.036259,"[""de"", ""en"", ""la""]","[0.969964, 0.005476, 0.004469]",4,1313.634644,de
1,de_de_001.wav,11.34,de,german,0.986359,0.018934,"[""de"", ""en"", ""nn""]","[0.986359, 0.004308, 0.001379]",3,1236.272461,de
2,de_de_002.wav,9.00,de,german,0.981416,0.025858,"[""de"", ""en"", ""nn""]","[0.981416, 0.009591, 0.00173]",3,1222.096680,de
3,de_de_003.wav,12.36,de,german,0.997502,0.001760,"[""de"", ""en"", ""nn""]","[0.997502, 0.001284, 0.000244]",4,1236.998169,de
4,de_de_004.wav,15.48,de,german,0.783887,0.164619,"[""de"", ""en"", ""ru""]","[0.783887, 0.045023, 0.040064]",4,1435.494019,de
5,de_de_005.wav,6.66,de,german,0.996603,0.000993,"[""de"", ""en"", ""nn""]","[0.996603, 0.00168, 0.000425]",2,1285.007324,de
6,en_us_000.wav,6.54,en,english,0.990313,0.007713,"[""en"", ""nn"", ""cy""]","[0.990313, 0.002546, 0.001781]",2,1549.418457,en
7,en_us_001.wav,16.38,en,english,0.978540,0.016878,"[""en"", ""ja"", ""nn""]","[0.97854, 0.006751, 0.002086]",5,1754.493530,en
8,en_us_002.wav,7.92,en,english,0.977062,0.025001,"[""en"", ""cy"", ""nn""]","[0.977062, 0.002696, 0.002657]",2,1141.110840,en
9,en_us_003.wav,4.14,en,english,0.815747,0.017108,"[""en"", ""hi"", ""haw""]","[0.815747, 0.037402, 0.02623]",2,1818.065186,en


,label,precision,recall,f1-score,support
0,de,1.0,1.0,1.0,2.0
1,en,1.0,1.0,1.0,2.0
2,ru,1.0,1.0,1.0,2.0
3,accuracy,1.0,1.0,1.0,1.0
4,macro avg,1.0,1.0,1.0,6.0
5,weighted avg,1.0,1.0,1.0,6.0


## Module B — TTS Evaluation
Сопоставление по `filename stem`, расчет метрик и графики сравнения систем.


In [5]:
from collections import defaultdict

def utt_id(p): return Path(p).stem

def index_by_id(root):
    d = {}; dup = 0
    for p in list_audio_files(root):
        u = utt_id(p)
        if u in d: dup += 1; continue
        d[u] = p
    if dup: print("dup ids in", root, dup)
    return d

def pitch_stats(y, sr):
    try:
        f0, vf, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr, frame_length=1024, hop_length=256)
        v = f0[~np.isnan(f0)]
        return (float(np.mean(v)), float(np.std(v)), float(np.mean(~np.isnan(f0)))) if len(v) else (np.nan,np.nan,0.0)
    except Exception:
        try:
            f0 = librosa.yin(y, fmin=50, fmax=500, sr=sr, frame_length=1024, hop_length=256); f0 = f0[np.isfinite(f0)]
            return (float(np.mean(f0)), float(np.std(f0)), np.nan) if len(f0) else (np.nan,np.nan,np.nan)
        except Exception: return np.nan,np.nan,np.nan

def tts_feats(y, sr):
    r = librosa.feature.rms(y=y, frame_length=1024, hop_length=256)[0]
    c = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    ro = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)[0]
    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    pm, ps, vr = pitch_stats(y, sr)
    return {
        "duration_sec": float(len(y)/sr),
        "rms_dbfs": float(20*np.log10(max(np.sqrt(np.mean(y**2)+1e-12),1e-8))),
        "peak_dbfs": float(20*np.log10(max(np.max(np.abs(y))+1e-12,1e-8))),
        "snr_proxy_db": float(20*np.log10((np.percentile(r,95)+1e-9)/(np.percentile(r,10)+1e-9))),
        "spectral_centroid_hz": float(np.mean(c)),
        "spectral_rolloff_hz": float(np.mean(ro)),
        "spectral_bandwidth_hz": float(np.mean(bw)),
        "pitch_mean_hz": pm,
        "pitch_std_hz": ps,
        "voiced_ratio": vr,
    }

def ref_scores(ref_p, deg_p, stoi_on=True, pesq_on=False):
    out = {}
    r, _ = load_audio(ref_p); d, _ = load_audio(deg_p); n = min(len(r), len(d))
    if n < int(0.25*CONFIG["sample_rate"]):
        if stoi_on: out["stoi"] = np.nan
        if pesq_on: out["pesq_wb"] = np.nan
        return out
    r, d = r[:n], d[:n]
    if stoi_on:
        try:
            from pystoi import stoi
            out["stoi"] = float(stoi(r, d, CONFIG["sample_rate"], extended=False))
        except Exception: out["stoi"] = np.nan
    if pesq_on:
        try:
            from pesq import pesq
            out["pesq_wb"] = float(pesq(CONFIG["sample_rate"], r, d, "wb"))
        except Exception: out["pesq_wb"] = np.nan
    return out

sys_ok = {k:Path(v) for k,v in CONFIG["tts"]["systems"].items() if Path(v).exists()}
if len(sys_ok) < 2:
    print("Need >=2 valid TTS folders:", CONFIG["tts"]["systems"])
    tts_df, tts_sum, tts_err, tts_stats = pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), {"error":"not enough systems"}
else:
    idx = {k:index_by_id(v) for k,v in sys_ok.items()}
    sets = {k:set(v.keys()) for k,v in idx.items()}
    common = set.intersection(*sets.values()) if sets else set(); union = set.union(*sets.values()) if sets else set()
    ref_idx = index_by_id(CONFIG["tts"]["reference_dir"]) if CONFIG["tts"].get("reference_dir") and Path(CONFIG["tts"]["reference_dir"]).exists() else {}
    rows, errs = [], []
    for u in tqdm(sorted(common), desc="TTS eval"):
        rp = ref_idx.get(u)
        for s in sys_ok.keys():
            p = idx[s][u]
            try:
                y, sr = load_audio(p); rec = {"utt_id":u, "system":s, "filename":Path(p).name}; rec.update(tts_feats(y, sr))
                if rp is not None: rec.update(ref_scores(rp, p, CONFIG["tts"]["compute_stoi"], CONFIG["tts"]["compute_pesq"]))
                else:
                    if CONFIG["tts"]["compute_stoi"]: rec["stoi"] = np.nan
                    if CONFIG["tts"]["compute_pesq"]: rec["pesq_wb"] = np.nan
                rows.append(rec)
            except Exception as e:
                errs.append({"utt_id":u, "system":s, "file":str(p), "error":str(e)})
    tts_df, tts_err = pd.DataFrame(rows), pd.DataFrame(errs)
    if not tts_df.empty:
        num = [c for c in tts_df.columns if c not in {"utt_id","system","filename"} and pd.api.types.is_numeric_dtype(tts_df[c])]
        tts_sum = tts_df.groupby("system")[num].agg(["mean","std","median"]).round(4)
    else:
        tts_sum = pd.DataFrame()
    tts_stats = {
        "system_file_counts": {k: len(v) for k,v in sets.items()},
        "common_ids": int(len(common)),
        "union_ids": int(len(union)),
        "missing_counts_vs_union": {k: int(len(union - v)) for k,v in sets.items()},
        "reference_available": bool(ref_idx),
        "evaluated_rows": int(len(tts_df)),
        "error_rows": int(len(tts_err)),
    }

tts_dir = P["results"] / "tts_eval"; tts_dir.mkdir(parents=True, exist_ok=True)
if not tts_df.empty:
    tts_df.to_csv(tts_dir/"tts_metrics.csv", index=False); ART["tables"].append(str(tts_dir/"tts_metrics.csv"))
if not tts_sum.empty:
    flat = tts_sum.copy(); flat.columns = [f"{a}_{b}" for a,b in flat.columns]; flat = flat.reset_index()
    flat.to_csv(tts_dir/"tts_summary.csv", index=False); ART["tables"].append(str(tts_dir/"tts_summary.csv"))
if not tts_err.empty:
    tts_err.to_csv(tts_dir/"tts_errors.csv", index=False); ART["tables"].append(str(tts_dir/"tts_errors.csv"))
(tts_dir/"tts_stats.json").write_text(json.dumps(tts_stats, ensure_ascii=False, indent=2), encoding="utf-8"); ART["tables"].append(str(tts_dir/"tts_stats.json"))

for m in CONFIG["tts"]["plot_metrics"]:
    if m not in tts_df.columns: continue
    d = tts_df[["system", m]].dropna();
    if d.empty: continue
    fig, ax = plt.subplots(figsize=(7,4)); sns.violinplot(data=d, x="system", y=m, inner="quartile", ax=ax, color="#8ecae6"); sns.stripplot(data=d, x="system", y=m, ax=ax, color="black", alpha=0.35, size=2)
    ax.set_title(f"TTS metric: {m}"); fig.tight_layout(); p = tts_dir/f"tts_{m}_violin.png"; fig.savefig(p, dpi=150); plt.close(fig); ART["plots"].append(str(p))
if {"pitch_std_hz","spectral_centroid_hz"}.issubset(set(tts_df.columns)):
    d = tts_df[["system","pitch_std_hz","spectral_centroid_hz"]].dropna()
    if not d.empty:
        fig, ax = plt.subplots(figsize=(6,5)); sns.scatterplot(data=d, x="pitch_std_hz", y="spectral_centroid_hz", hue="system", ax=ax)
        ax.set_title("Pitch variation vs spectral centroid"); fig.tight_layout(); p = tts_dir/"tts_scatter_pitchstd_vs_centroid.png"; fig.savefig(p, dpi=150); plt.close(fig); ART["plots"].append(str(p))

print("TTS stats:", tts_stats)
if not tts_df.empty: display(tts_df.head(20))
if not tts_sum.empty:
    flat = tts_sum.copy(); flat.columns = [f"{a}_{b}" for a,b in flat.columns]; display(flat.reset_index().head(20))


Need >=2 valid TTS folders: {'tts_system_A': '/content/speech_toolkit/tts_systems/tts_system_A', 'tts_system_B': '/content/speech_toolkit/tts_systems/tts_system_B'}
TTS stats: {'error': 'not enough systems'}


## Module C — Loudness Normalization
RMS/LUFS нормализация, простая компрессия/гейт, peak protection, сравнение до/после.


In [6]:
try:
    import pyloudnorm as pyln
    HAS_LN = True
except Exception:
    HAS_LN = False
METER = {}

def meter(sr):
    if not HAS_LN: return None
    if sr not in METER: METER[sr] = pyln.Meter(sr)
    return METER[sr]

def profile(y, sr):
    eps = 1e-9
    rms = float(np.sqrt(np.mean(y**2)+eps)); peak = float(np.max(np.abs(y))+eps)
    rdb = float(20*np.log10(max(rms,eps))); pdb = float(20*np.log10(max(peak,eps)))
    fr = librosa.feature.rms(y=y, frame_length=1024, hop_length=256)[0] + eps
    dyn = float(np.percentile(20*np.log10(fr),95) - np.percentile(20*np.log10(fr),10))
    lufs = np.nan
    if HAS_LN:
        try: lufs = float(meter(sr).integrated_loudness(y))
        except Exception: lufs = np.nan
    return {"rms_dbfs":rdb, "peak_dbfs":pdb, "crest_db": float(pdb-rdb), "dynamic_range_db":dyn, "lufs":lufs}

def frame_gain(y, sr, thr_db, ratio, mode="comp", frame_ms=20, hop_ms=10):
    fl = max(int(sr*frame_ms/1000),256); hl = max(int(sr*hop_ms/1000),128)
    fr = librosa.feature.rms(y=y, frame_length=fl, hop_length=hl, center=True)[0]
    db = 20*np.log10(np.maximum(fr,1e-8))
    if mode == "comp":
        over = np.maximum(db-thr_db,0.0); gdb = -over*(1.0-1.0/max(ratio,1.01))
    else:
        below = np.maximum(thr_db-db,0.0); gdb = -below*max(ratio-1.0,0.0)
    g = 10**(gdb/20.0)
    pos = np.arange(len(g))*hl
    if len(pos)==0: return y
    if pos[-1] < len(y)-1: pos = np.append(pos, len(y)-1); g = np.append(g, g[-1])
    return (y*np.interp(np.arange(len(y)), pos, g).astype(np.float32)).astype(np.float32)

def norm_rms(y, target_dbfs):
    cur = float(20*np.log10(max(np.sqrt(np.mean(y**2)+1e-12),1e-8)))
    return (y*(10**((target_dbfs-cur)/20.0))).astype(np.float32)

def norm_lufs(y, sr, target_lufs):
    if not HAS_LN: return y
    try:
        L = meter(sr).integrated_loudness(y)
        return pyln.normalize.loudness(y, L, target_lufs).astype(np.float32)
    except Exception:
        return y

def peak_limit(y, peak_dbfs=-1.0):
    lim = 10**(peak_dbfs/20.0); p = np.max(np.abs(y))
    if p > lim and p > 0: y = y*(lim/p)
    return np.clip(y,-1.0,1.0).astype(np.float32)

def process_one(y, sr, cfg):
    z = y.astype(np.float32)
    if cfg.get("use_gate",False): z = frame_gain(z, sr, cfg.get("gate_thr_db",-50), cfg.get("gate_ratio",2), mode="gate")
    if cfg.get("use_comp",True): z = frame_gain(z, sr, cfg.get("comp_thr_db",-22), cfg.get("comp_ratio",3), mode="comp")
    z = norm_lufs(z, sr, cfg.get("target_lufs",-16)) if (cfg.get("use_lufs",True) and HAS_LN) else norm_rms(z, cfg.get("target_dbfs",-20))
    return peak_limit(z, cfg.get("peak_limit_dbfs",-1.0))

def infer_lang_from_filename(filename):
    stem = Path(filename).stem.lower()
    for sep in ("-", ".", " "):
        stem = stem.replace(sep, "_")
    parts = [x for x in stem.split("_") if x]
    if parts and len(parts[0]) == 2 and parts[0].isalpha():
        return parts[0]
    return "unknown"


def build_language_lookup():
    lookup = {}

    meta_csv = CONFIG.get("data", {}).get("metadata_csv", "")
    if meta_csv and Path(meta_csv).exists():
        try:
            meta = pd.read_csv(meta_csv)
            if {"filename", "true_lang"}.issubset(meta.columns):
                for _, row in meta.iterrows():
                    fn = str(Path(str(row["filename"])).name)
                    lookup[fn] = str(row["true_lang"])
        except Exception as e:
            print("[WARN] Language mapping from metadata failed:", e)

    lid_table = globals().get("lid_df", pd.DataFrame())
    if isinstance(lid_table, pd.DataFrame) and {"filename", "pred_lang"}.issubset(lid_table.columns):
        for _, row in lid_table.iterrows():
            fn = str(Path(str(row["filename"])).name)
            if fn not in lookup or lookup[fn] in (None, "", "nan"):
                lookup[fn] = str(row["pred_lang"])

    return lookup


def run_norm(inp, out, cfg, lang_lookup=None):
    inp, out = Path(inp), Path(out); out.mkdir(parents=True, exist_ok=True)
    rows, errs = [], []
    lang_lookup = lang_lookup or {}
    for p in tqdm(list_audio_files(inp), desc="Normalize"):
        try:
            y, sr = load_audio(p)
            b = profile(y, sr)
            z = process_one(y, sr, cfg)
            a = profile(z, sr)
            rel = p.relative_to(inp); q = (out/rel).with_suffix(".wav"); q.parent.mkdir(parents=True, exist_ok=True)
            sf.write(q, z, sr)
            lang = lang_lookup.get(p.name, infer_lang_from_filename(p.name))
            rows.append({"filename":p.name, "lang":lang, "input_path":str(p), "output_path":str(q), **{f"before_{k}":v for k,v in b.items()}, **{f"after_{k}":v for k,v in a.items()}})
        except Exception as e:
            errs.append({"file":str(p), "error":str(e)})
    return pd.DataFrame(rows), pd.DataFrame(errs)

def norm_plots(df, out):
    out = Path(out); out.mkdir(parents=True, exist_ok=True); pp = []
    for m in ["rms_dbfs","peak_dbfs","lufs","dynamic_range_db","crest_db"]:
        b, a = f"before_{m}", f"after_{m}"
        if b not in df.columns or a not in df.columns: continue
        d = pd.concat([pd.DataFrame({"stage":"before","value":pd.to_numeric(df[b],errors='coerce')}), pd.DataFrame({"stage":"after","value":pd.to_numeric(df[a],errors='coerce')})], ignore_index=True).dropna()
        if d.empty: continue
        fig, ax = plt.subplots(figsize=(6,4)); sns.boxplot(data=d, x="stage", y="value", ax=ax, palette="Set2"); sns.stripplot(data=d, x="stage", y="value", ax=ax, color="black", alpha=0.35, size=2)
        ax.set_title(f"{m}: before vs after"); fig.tight_layout(); p = out/f"norm_{m}_before_after.png"; fig.savefig(p, dpi=150); plt.close(fig); pp.append(str(p))
    return pp

def preview(df, n=1, by_language=True):
    if df.empty: return
    n = max(int(n), 1)

    if by_language and "lang" in df.columns:
        d = df.copy()
        d["lang"] = d["lang"].fillna("unknown").astype(str)
        for lang in sorted(d["lang"].unique()):
            sub = d[d["lang"] == lang].head(n)
            if sub.empty: continue
            display(Markdown(f"### Language: `{lang}`"))
            for _, r in sub.iterrows():
                display(Markdown(f"**{r['filename']}**"))
                display(Markdown("Before"))
                display(Audio(filename=r["input_path"]))
                display(Markdown("After"))
                display(Audio(filename=r["output_path"]))
    else:
        for _, r in df.head(n).iterrows():
            display(Markdown(f"**{r['filename']}**"))
            display(Markdown("Before"))
            display(Audio(filename=r["input_path"]))
            display(Markdown("After"))
            display(Audio(filename=r["output_path"]))

norm_dir = P["results"] / "normalization"; norm_dir.mkdir(parents=True, exist_ok=True)
lang_lookup = build_language_lookup()
norm_df, norm_err = run_norm(CONFIG["norm"]["input_dir"], CONFIG["norm"]["output_dir"], CONFIG["norm"], lang_lookup=lang_lookup)
if not norm_df.empty:
    norm_df.to_csv(norm_dir/"normalization_metrics.csv", index=False); ART["tables"].append(str(norm_dir/"normalization_metrics.csv"))
if not norm_err.empty:
    norm_err.to_csv(norm_dir/"normalization_errors.csv", index=False); ART["tables"].append(str(norm_dir/"normalization_errors.csv"))
for p in norm_plots(norm_df, norm_dir/"plots"): ART["plots"].append(p)
print("normalized:", len(norm_df), "errors:", len(norm_err))
if not norm_df.empty: display(norm_df.head(20))
if CONFIG["norm"].get("preview_n",0) > 0:
    preview(norm_df, CONFIG["norm"]["preview_n"], by_language=CONFIG["norm"].get("preview_by_language", True))


Normalize:   0%|          | 0/18 [00:00<?, ?it/s]

normalized: 18 errors: 0


,filename,lang,input_path,output_path,before_rms_dbfs,before_peak_dbfs,before_crest_db,before_dynamic_range_db,before_lufs,after_rms_dbfs,after_peak_dbfs,after_crest_db,after_dynamic_range_db,after_lufs
0,de_de_000.wav,de,/content/speech_toolkit/input_audio/demo_lid/d...,/content/speech_toolkit/normalized/de_de_000.wav,-34.542114,-15.437236,19.104879,38.493103,-34.589884,-20.104892,-1.000001,19.104891,38.493122,-20.152650
1,de_de_001.wav,de,/content/speech_toolkit/input_audio/demo_lid/d...,/content/speech_toolkit/normalized/de_de_001.wav,-26.009203,-7.499466,18.509737,46.436462,-24.278600,-18.002384,-1.000000,17.002384,44.390930,-16.256859
2,de_de_002.wav,de,/content/speech_toolkit/input_audio/demo_lid/d...,/content/speech_toolkit/normalized/de_de_002.wav,-26.450300,-4.856388,21.593912,45.255920,-24.820411,-19.168847,-1.000000,18.168846,43.643532,-17.655853
3,de_de_003.wav,de,/content/speech_toolkit/input_audio/demo_lid/d...,/content/speech_toolkit/normalized/de_de_003.wav,-28.885268,-11.901314,16.983954,37.395500,-29.431784,-17.634062,-1.000000,16.634061,37.295322,-18.183933
4,de_de_004.wav,de,/content/speech_toolkit/input_audio/demo_lid/d...,/content/speech_toolkit/normalized/de_de_004.wav,-41.969504,-24.137710,17.831794,26.944958,-40.624429,-18.831862,-1.000000,17.831862,26.944969,-17.486719
5,de_de_005.wav,de,/content/speech_toolkit/input_audio/demo_lid/d...,/content/speech_toolkit/normalized/de_de_005.wav,-31.550558,-13.291175,18.259383,33.132618,-32.482432,-19.259389,-1.000000,18.259388,33.132618,-20.191257
6,en_us_000.wav,en,/content/speech_toolkit/input_audio/demo_lid/e...,/content/speech_toolkit/normalized/en_us_000.wav,-49.896696,-32.534179,17.362517,24.962971,-49.442732,-18.362940,-1.000000,17.362940,24.962984,-17.908553
7,en_us_001.wav,en,/content/speech_toolkit/input_audio/demo_lid/e...,/content/speech_toolkit/normalized/en_us_001.wav,-58.965733,-38.377056,20.588677,27.298134,-58.728363,-21.592100,-1.000000,20.592100,27.298218,-21.378519
8,en_us_002.wav,en,/content/speech_toolkit/input_audio/demo_lid/e...,/content/speech_toolkit/normalized/en_us_002.wav,-29.329202,-11.898184,17.431017,33.207272,-28.807228,-17.549294,-1.000000,16.549294,33.088810,-17.029552
9,en_us_003.wav,en,/content/speech_toolkit/input_audio/demo_lid/e...,/content/speech_toolkit/normalized/en_us_003.wav,-63.437924,-44.918738,18.519185,22.105263,-62.919778,-19.528779,-1.000000,18.528779,22.105335,-19.117591


### Language: `de`

**de_de_000.wav**

...

After

### Language: `ru`

**ru_ru_000.wav**

Before

After

## Отчет и экспорт
Сохранение таблиц (CSV), графиков (PNG) и итогового markdown/html отчета.


In [7]:
import markdown as mdlib

def flat(df):
    if df is None or df.empty: return pd.DataFrame()
    x = df.copy()
    if isinstance(x.columns, pd.MultiIndex): x.columns = [f"{a}_{b}" for a,b in x.columns]
    return x.reset_index()

def md_table(df, n=20):
    if df is None or len(df)==0: return "_No data_"
    s = df.head(n)
    try: return s.to_markdown(index=False)
    except Exception: return "```\n"+s.to_string(index=False)+"\n```"

lid_df = globals().get("lid_df", pd.DataFrame()); lid_err = globals().get("lid_err", pd.DataFrame()); lid_eval = globals().get("lid_eval", {})
tts_df = globals().get("tts_df", pd.DataFrame()); tts_sum = globals().get("tts_sum", pd.DataFrame()); tts_err = globals().get("tts_err", pd.DataFrame()); tts_stats = globals().get("tts_stats", {})
norm_df = globals().get("norm_df", pd.DataFrame()); norm_err = globals().get("norm_err", pd.DataFrame())

lines = []
lines += ["# Speech Toolkit Report", f"Generated (UTC): {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%SZ')}"]
lines += ["## Configuration", "```json", json.dumps(CONFIG, ensure_ascii=False, indent=2), "```"]
lines += ["## Module A: LID", f"- files: {len(lid_df)}", f"- errors: {len(lid_err)}"]
if lid_eval: lines += [f"- accuracy: {lid_eval.get('accuracy', float('nan')):.4f}", f"- f1_macro: {lid_eval.get('f1_macro', float('nan')):.4f}"]
cols = [c for c in ["filename","pred_lang","confidence","topk_langs","topk_probs","true_lang"] if c in lid_df.columns]
lines += ["LID preview:", md_table(lid_df[cols] if cols else lid_df)]
lines += ["## Module B: TTS Eval", f"- metric rows: {len(tts_df)}", f"- errors: {len(tts_err)}", "```json", json.dumps(tts_stats, ensure_ascii=False, indent=2), "```"]
if not tts_sum.empty: lines += ["TTS summary:", md_table(flat(tts_sum))]
lines += ["## Module C: Loudness Norm", f"- processed: {len(norm_df)}", f"- errors: {len(norm_err)}"]
if not norm_df.empty:
    sc = [c for c in ["before_rms_dbfs","after_rms_dbfs","before_peak_dbfs","after_peak_dbfs","before_lufs","after_lufs"] if c in norm_df.columns]
    if sc: lines += ["Normalization means:", md_table(norm_df[sc].mean(numeric_only=True).to_frame().T, n=1)]
lines += ["## Artifacts"]
for k in ["tables","plots","reports"]:
    lines += [f"### {k.capitalize()}"]
    lines += [f"- `{p}`" for p in ART.get(k,[])] if ART.get(k,[]) else ["- (none)"]

text = "\n\n".join(lines)
rep = P["results"] / "report"; rep.mkdir(parents=True, exist_ok=True)
md_path = rep / "speech_toolkit_report.md"; md_path.write_text(text, encoding="utf-8")
try: body = mdlib.markdown(text, extensions=["tables","fenced_code"])
except Exception: body = "<pre>"+text.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")+"</pre>"
html_path = rep / "speech_toolkit_report.html"
html_path.write_text(f"<!doctype html><html><head><meta charset='utf-8'><title>Speech Toolkit Report</title></head><body>{body}</body></html>", encoding="utf-8")
ART["reports"].extend([str(md_path), str(html_path)])
(rep/"artifacts_manifest.json").write_text(json.dumps(ART, ensure_ascii=False, indent=2), encoding="utf-8")
print("report:", md_path)
print("html:", html_path)
for f in sorted((P["results"]).rglob("*")):
    if f.is_file(): print(f)


report: /content/speech_toolkit/results/report/speech_toolkit_report.md
html: /content/speech_toolkit/results/report/speech_toolkit_report.html
/content/speech_toolkit/results/lid/lid_confidence.png
/content/speech_toolkit/results/lid/lid_confusion_matrix.png
/content/speech_toolkit/results/lid/lid_eval_metrics.json
/content/speech_toolkit/results/lid/lid_results.csv
/content/speech_toolkit/results/lid/mfcc_baseline_report.csv
/content/speech_toolkit/results/normalization/normalization_metrics.csv
/content/speech_toolkit/results/normalization/plots/norm_crest_db_before_after.png
/content/speech_toolkit/results/normalization/plots/norm_dynamic_range_db_before_after.png
/content/speech_toolkit/results/normalization/plots/norm_lufs_before_after.png
/content/speech_toolkit/results/normalization/plots/norm_peak_dbfs_before_after.png
/content/speech_toolkit/results/normalization/plots/norm_rms_dbfs_before_after.png
/content/speech_toolkit/results/report/artifacts_manifest.json
/content/speec